In [0]:
CREATE OR REPLACE TABLE production.supply_analytics.option_price_discrepancy AS

WITH pricing_base AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    from_price_without_deal,
    currency_iso_code,
    individual_categories,
    group_categories,
    has_individual_pricing,
    has_group_pricing
  FROM production.supply_analytics.pricing_feature_audit_base
  WHERE uses_external_pricing = 0
    AND from_price_without_deal IS NOT NULL
    AND from_price_without_deal > 0
),

-- GET LATEST EXCHANGE RATES
latest_exchange_rates AS (
  SELECT 
    c.iso_code,
    er.exchange_rate
  FROM production.dwh.dim_currency c
  INNER JOIN production.dwh.dim_currency_all_dates er
    ON c.currency_id = er.currency_id
  WHERE er.update_timestamp = CURRENT_DATE()
),

pricing_with_fx AS (
  SELECT 
    pb.*,
    COALESCE(ler.exchange_rate, 1.0) as exchange_rate,
    pb.from_price_without_deal / COALESCE(ler.exchange_rate, 1.0) as from_price_eur
  FROM pricing_base pb
  LEFT JOIN latest_exchange_rates ler
    ON pb.currency_iso_code = ler.iso_code
),

-- GET BASE RETAIL PRICE - ADULT CATEGORY ONLY FOR INDIVIDUAL PRICING
base_retail_price_individual AS (
  SELECT 
    tour_option_id,
    pricing_id,
    MIN(price_tier.retailPrice) as base_retail_price,
    MIN(price_tier.retailPrice / exchange_rate) as base_retail_price_eur
  FROM pricing_with_fx pb
  LATERAL VIEW OUTER EXPLODE(pb.individual_categories) AS ind_cat
  LATERAL VIEW OUTER EXPLODE(ind_cat.prices) AS price_tier
  WHERE pb.has_individual_pricing = 1
    AND UPPER(ind_cat.ticketCategory) = 'ADULT'
  GROUP BY tour_option_id, pricing_id
),

-- GET BASE RETAIL PRICE - SMALLEST GROUP PRICE (BASE GROUPS ONLY, NOT ADDITIONAL PAX)
base_retail_price_group AS (
  SELECT 
    tour_option_id,
    pricing_id,
    MIN(price_tier.retailPrice) as base_retail_price,
    MIN(price_tier.retailPrice / exchange_rate) as base_retail_price_eur
  FROM pricing_with_fx pb
  LATERAL VIEW OUTER EXPLODE(pb.group_categories) AS grp_cat
  LATERAL VIEW OUTER EXPLODE(grp_cat.prices) AS price_tier
  WHERE pb.has_group_pricing = 1
    AND grp_cat.isAdditionalPaxForGroup = false
  GROUP BY tour_option_id, pricing_id
),

-- COMBINE RETAIL PRICES - PRIORITIZE INDIVIDUAL (ADULT) OVER GROUP
option_prices AS (
  SELECT 
    pw.supplier_id,
    pw.segment,
    pw.tour_id,
    pw.tour_option_id,
    pw.pricing_id,
    pw.tour_title,
    pw.tour_option_title,
    pw.location_name,
    pw.activity_type,
    pw.tour_category,
    pw.currency_iso_code,
    pw.exchange_rate,
    pw.from_price_without_deal,
    pw.from_price_eur,
    pw.has_individual_pricing,
    pw.has_group_pricing,
    
    -- Use individual (ADULT) if available, otherwise group
    COALESCE(brpi.base_retail_price, brpg.base_retail_price) as base_retail_price,
    COALESCE(brpi.base_retail_price_eur, brpg.base_retail_price_eur) as base_retail_price_eur,
    
    CASE 
      WHEN brpi.base_retail_price IS NOT NULL THEN 'ADULT_INDIVIDUAL'
      WHEN brpg.base_retail_price IS NOT NULL THEN 'SMALLEST_GROUP'
      ELSE NULL
    END as pricing_type_used
    
  FROM pricing_with_fx pw
  LEFT JOIN base_retail_price_individual brpi 
    ON pw.tour_option_id = brpi.tour_option_id AND pw.pricing_id = brpi.pricing_id
  LEFT JOIN base_retail_price_group brpg 
    ON pw.tour_option_id = brpg.tour_option_id AND pw.pricing_id = brpg.pricing_id
),

-- CALCULATE GLOBAL BENCHMARKS
global_benchmarks AS (
  SELECT 
    PERCENTILE(from_price_eur, 0.50) as global_median_eur,
    PERCENTILE(from_price_eur, 0.75) as global_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as global_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as global_p95_eur,
    PERCENTILE(from_price_eur, 0.99) as global_p99_eur,
    AVG(from_price_eur) as global_avg_eur,
    STDDEV(from_price_eur) as global_stddev_eur
  FROM option_prices
),

-- CALCULATE LOCATION BENCHMARKS
location_benchmarks AS (
  SELECT 
    location_name,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as location_median_eur,
    PERCENTILE(from_price_eur, 0.75) as location_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as location_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as location_p95_eur,
    AVG(from_price_eur) as location_avg_eur,
    STDDEV(from_price_eur) as location_stddev_eur
  FROM option_prices
  GROUP BY location_name
  HAVING COUNT(DISTINCT tour_id) >= 10
),

-- CALCULATE CATEGORY BENCHMARKS
category_benchmarks AS (
  SELECT 
    tour_category,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as category_median_eur,
    PERCENTILE(from_price_eur, 0.75) as category_p75_eur,
    PERCENTILE(from_price_eur, 0.90) as category_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as category_p95_eur,
    AVG(from_price_eur) as category_avg_eur,
    STDDEV(from_price_eur) as category_stddev_eur
  FROM option_prices
  GROUP BY tour_category
  HAVING COUNT(DISTINCT tour_id) >= 10
),

-- CALCULATE LOCATION + CATEGORY BENCHMARKS
location_category_benchmarks AS (
  SELECT 
    location_name,
    tour_category,
    COUNT(DISTINCT tour_id) as num_tours,
    PERCENTILE(from_price_eur, 0.50) as loc_cat_median_eur,
    PERCENTILE(from_price_eur, 0.90) as loc_cat_p90_eur,
    PERCENTILE(from_price_eur, 0.95) as loc_cat_p95_eur,
    AVG(from_price_eur) as loc_cat_avg_eur
  FROM option_prices
  GROUP BY location_name, tour_category
  HAVING COUNT(DISTINCT tour_id) >= 5
),

-- TOUR-LEVEL AGGREGATES
tour_aggregates AS (
  SELECT 
    tour_id,
    COUNT(DISTINCT tour_option_id) as num_options,
    COUNT(DISTINCT from_price_eur) as num_distinct_prices,
    MIN(from_price_eur) as tour_min_price_eur,
    MAX(from_price_eur) as tour_max_price_eur,
    AVG(from_price_eur) as tour_avg_price_eur,
    STDDEV(from_price_eur) as tour_stddev_price_eur,
    CASE 
      WHEN AVG(from_price_eur) > 0 
      THEN STDDEV(from_price_eur) / AVG(from_price_eur)
      ELSE 0
    END as tour_price_cv,
    CASE 
      WHEN MIN(from_price_eur) > 0 
      THEN (MAX(from_price_eur) - MIN(from_price_eur)) / MIN(from_price_eur)
      ELSE 0
    END as tour_price_spread_ratio
  FROM option_prices
  GROUP BY tour_id
),

-- COMBINE ALL DATA
options_with_benchmarks AS (
  SELECT 
    op.*,
    gb.global_median_eur,
    gb.global_p90_eur,
    gb.global_p95_eur,
    gb.global_p99_eur,
    gb.global_avg_eur,
    gb.global_stddev_eur,
    lb.location_median_eur,
    lb.location_p90_eur,
    lb.location_p95_eur,
    lb.location_avg_eur,
    cb.category_median_eur,
    cb.category_p90_eur,
    cb.category_p95_eur,
    cb.category_avg_eur,
    lcb.loc_cat_median_eur,
    lcb.loc_cat_p90_eur,
    lcb.loc_cat_p95_eur,
    ta.num_options,
    ta.num_distinct_prices,
    ta.tour_min_price_eur,
    ta.tour_max_price_eur,
    ta.tour_avg_price_eur,
    ta.tour_price_cv,
    ta.tour_price_spread_ratio
  FROM option_prices op
  CROSS JOIN global_benchmarks gb
  LEFT JOIN location_benchmarks lb ON op.location_name = lb.location_name
  LEFT JOIN category_benchmarks cb ON op.tour_category = cb.tour_category
  LEFT JOIN location_category_benchmarks lcb 
    ON op.location_name = lcb.location_name AND op.tour_category = lcb.tour_category
  LEFT JOIN tour_aggregates ta ON op.tour_id = ta.tour_id
),

-- BUCKET 1: ABNORMALLY HIGH FROM PRICES
bucket1_high_from_prices AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    currency_iso_code,
    from_price_without_deal,
    from_price_eur,
    exchange_rate,
    has_individual_pricing,
    has_group_pricing,
    
    'BUCKET_1_HIGH_FROM_PRICE' as issue_bucket,
    
    CASE 
      WHEN loc_cat_p95_eur IS NOT NULL AND from_price_eur > loc_cat_p95_eur * 2 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / loc_cat_p95_eur - 1), 0), 
                  '% above P95 for ', location_name, ' - ', tour_category, ' (P95: ', ROUND(loc_cat_p95_eur, 2), ' EUR)')
      WHEN location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur * 2 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / location_p95_eur - 1), 0), 
                  '% above P95 for ', location_name, ' (P95: ', ROUND(location_p95_eur, 2), ' EUR)')
      WHEN category_p95_eur IS NOT NULL AND from_price_eur > category_p95_eur * 2 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is ', 
                  ROUND(100.0 * (from_price_eur / category_p95_eur - 1), 0), 
                  '% above P95 for ', tour_category, ' (P95: ', ROUND(category_p95_eur, 2), ' EUR)')
      WHEN from_price_eur > global_p99_eur 
      THEN CONCAT('From price ', ROUND(from_price_eur, 2), ' EUR is above global P99 (', 
                  ROUND(global_p99_eur, 2), ' EUR)')
      ELSE NULL
    END as problem_description,
    
    CASE 
      WHEN loc_cat_p95_eur IS NOT NULL AND from_price_eur > loc_cat_p95_eur * 2 THEN 10
      WHEN location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur * 2 THEN 9
      WHEN category_p95_eur IS NOT NULL AND from_price_eur > category_p95_eur * 2 THEN 8
      WHEN from_price_eur > global_p99_eur THEN 7
      ELSE 0
    END as severity_score,
    
    COALESCE(loc_cat_median_eur, location_median_eur, category_median_eur, global_median_eur) as benchmark_median_eur,
    COALESCE(loc_cat_p95_eur, location_p95_eur, category_p95_eur, global_p95_eur) as benchmark_p95_eur,
    
    NULL as pricing_type_used,
    NULL as tour_min_price_eur,
    NULL as tour_max_price_eur,
    NULL as tour_price_spread_pct,
    NULL as base_retail_price,
    NULL as base_retail_price_eur,
    NULL as price_gap_pct
    
  FROM options_with_benchmarks
  WHERE 
    (loc_cat_p95_eur IS NOT NULL AND from_price_eur > loc_cat_p95_eur * 2)
    OR (location_p95_eur IS NOT NULL AND from_price_eur > location_p95_eur * 2)
    OR (category_p95_eur IS NOT NULL AND from_price_eur > category_p95_eur * 2)
    OR from_price_eur > global_p99_eur
),

-- BUCKET 2: HIGH VARIANCE WITHIN TOUR
bucket2_tour_variance AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    currency_iso_code,
    from_price_without_deal,
    from_price_eur,
    exchange_rate,
    has_individual_pricing,
    has_group_pricing,
    
    'BUCKET_2_TOUR_VARIANCE' as issue_bucket,
    
    CONCAT('Tour has ', num_options, ' options with ', 
           ROUND(tour_price_spread_ratio * 100, 0), '% price spread (CV: ',
           ROUND(tour_price_cv * 100, 1), '%). This option: ', ROUND(from_price_eur, 2), 
           ' EUR vs tour range ', ROUND(tour_min_price_eur, 2), '-', ROUND(tour_max_price_eur, 2), ' EUR') as problem_description,
    
    CASE 
      WHEN tour_price_spread_ratio > 2.0 THEN 10
      WHEN tour_price_spread_ratio > 1.0 THEN 8
      WHEN tour_price_spread_ratio > 0.5 THEN 6
      ELSE 4
    END as severity_score,
    
    NULL as benchmark_median_eur,
    NULL as benchmark_p95_eur,
    NULL as pricing_type_used,
    
    tour_min_price_eur,
    tour_max_price_eur,
    ROUND(tour_price_spread_ratio * 100, 0) as tour_price_spread_pct,
    NULL as base_retail_price,
    NULL as base_retail_price_eur,
    NULL as price_gap_pct
    
  FROM options_with_benchmarks
  WHERE num_options > 1 
    AND tour_price_spread_ratio > 0.3
),

-- BUCKET 3: FROM PRICE VS RETAIL PRICE MISMATCH
bucket3_price_mismatch AS (
  SELECT 
    supplier_id,
    segment,
    tour_id,
    tour_option_id,
    pricing_id,
    tour_title,
    tour_option_title,
    location_name,
    activity_type,
    tour_category,
    currency_iso_code,
    from_price_without_deal,
    from_price_eur,
    exchange_rate,
    has_individual_pricing,
    has_group_pricing,
    
    'BUCKET_3_PRICE_MISMATCH' as issue_bucket,
    
    CASE 
      WHEN from_price_eur > base_retail_price_eur * 1.05 
      THEN CONCAT('From price (', ROUND(from_price_without_deal, 0), ' ', currency_iso_code, 
                  ') is ', ROUND(100.0 * (from_price_eur - base_retail_price_eur) / base_retail_price_eur, 1),
                  '% HIGHER than ', pricing_type_used, ' retail price (', ROUND(base_retail_price, 0), ' ', currency_iso_code, ')')
      WHEN from_price_eur < base_retail_price_eur * 0.95 
      THEN CONCAT('From price (', ROUND(from_price_without_deal, 0), ' ', currency_iso_code, 
                  ') is ', ROUND(100.0 * (base_retail_price_eur - from_price_eur) / base_retail_price_eur, 1),
                  '% LOWER than ', pricing_type_used, ' retail price (', ROUND(base_retail_price, 0), ' ', currency_iso_code, ')')
      ELSE NULL
    END as problem_description,
    
    CASE 
      WHEN from_price_eur > base_retail_price_eur * 1.05 THEN 10
      WHEN from_price_eur < base_retail_price_eur * 0.95 THEN 8
      ELSE 0
    END as severity_score,
    
    NULL as benchmark_median_eur,
    NULL as benchmark_p95_eur,
    pricing_type_used,
    NULL as tour_min_price_eur,
    NULL as tour_max_price_eur,
    NULL as tour_price_spread_pct,
    
    base_retail_price,
    base_retail_price_eur,
    ROUND(100.0 * (from_price_eur - base_retail_price_eur) / base_retail_price_eur, 1) as price_gap_pct
    
  FROM options_with_benchmarks
  WHERE base_retail_price_eur IS NOT NULL
    AND (from_price_eur > base_retail_price_eur * 1.05 
         OR from_price_eur < base_retail_price_eur * 0.95)
),

-- COMBINE ALL BUCKETS
all_issues AS (
  SELECT * FROM bucket1_high_from_prices
  UNION ALL
  SELECT * FROM bucket2_tour_variance
  UNION ALL
  SELECT * FROM bucket3_price_mismatch
)

-- FINAL OUTPUT
SELECT 
  issue_bucket,
  severity_score,
  supplier_id,
  segment,
  tour_id,
  tour_option_id,
  pricing_id,
  tour_title,
  tour_option_title,
  location_name,
  activity_type,
  tour_category,
  has_individual_pricing,
  has_group_pricing,
  pricing_type_used,
  problem_description,
  
  currency_iso_code,
  from_price_without_deal as current_from_price,
  ROUND(from_price_eur, 2) as from_price_eur,
  ROUND(exchange_rate, 4) as exchange_rate_to_eur,
  
  -- Bucket-specific fields
  ROUND(benchmark_median_eur, 2) as benchmark_median_eur,
  ROUND(benchmark_p95_eur, 2) as benchmark_p95_eur,
  ROUND(tour_min_price_eur, 2) as tour_min_from_price_eur,
  ROUND(tour_max_price_eur, 2) as tour_max_from_price_eur,
  tour_price_spread_pct,
  base_retail_price,
  ROUND(base_retail_price_eur, 2) as base_retail_price_eur,
  price_gap_pct
  
FROM all_issues
WHERE problem_description IS NOT NULL
ORDER BY issue_bucket, severity_score DESC, tour_id, tour_option_id;
